# End-to-End Android APK Vulnerability Scanner Pipeline
This notebook automatically orchestrates `jadx` decompilation, AST parsing with `tree-sitter`, Data Flow Graph generation, and GraphCodeBERT inference natively in a Kaggle environment for an entire directory of APKs.

In [ ]:
!pip install torch transformers tree_sitter==0.21.3 androguard==3.3.3 tqdm -q


In [ ]:
import os
import shutil
import urllib.request
import zipfile
import subprocess
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from transformers import RobertaConfig, RobertaModel, AutoTokenizer
from tree_sitter import Language, Parser

# --- 1. DFG UTILS ---
def setup_tree_sitter():
    build_dir = 'build'
    lib_path = os.path.join(build_dir, 'my-languages.so')
    java_repo = 'tree-sitter-java'

    if not os.path.exists(lib_path):
        os.makedirs(build_dir, exist_ok=True)
        if not os.path.exists(java_repo):
            print(f"Cloning {java_repo}...")
            os.system(f"git clone https://github.com/tree-sitter/tree-sitter-java")
        
        print("Compiling tree-sitter languages...")
        Language.build_library(lib_path, [java_repo])
        print("Compilation complete.")

    try:
        java_lang = Language(lib_path, 'java')
        parser = Parser()
        parser.set_language(java_lang)
        return java_lang, parser
    except Exception as e:
        print(f"CRITICAL ERROR during parser setup: {e}")
        return None, None

def tree_to_token_index(root_node):
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        return [(root_node.start_point, root_node.end_point)]
    else:
        code_tokens = []
        for child in root_node.children:
            code_tokens += tree_to_token_index(child)
        return code_tokens

def tree_to_variable_index(root_node, index_to_code):
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        index = (root_node.start_point, root_node.end_point)
        _, code = index_to_code.get(index, (None, root_node.type))
        if root_node.type != code:
            return [(root_node.start_point, root_node.end_point)]
        else:
            return []
    else:
        code_tokens = []
        for child in root_node.children:
            code_tokens += tree_to_variable_index(child, index_to_code)
        return code_tokens

def index_to_code_token(index, code):
    start_point = index[0]
    end_point = index[1]
    if start_point[0] == end_point[0]:
        s = code[start_point[0]][start_point[1]:end_point[1]]
    else:
        s = code[start_point[0]][0:0]
        s += code[start_point[0]][start_point[1]:]
        for i in range(start_point[0] + 1, end_point[0]):
            s += code[i]
        s += code[end_point[0]][:end_point[1]]
    return s

def DFG_java(root_node, index_to_code, states):
    assignment = ['assignment_expression']
    def_statement = ['variable_declarator']
    increment_statement = ['update_expression']
    if_statement = ['if_statement', 'else']
    for_statement = ['for_statement']
    enhanced_for_statement = ['enhanced_for_statement']
    while_statement = ['while_statement']
    do_first_statement = []
    
    states = states.copy()
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        idx, code = index_to_code.get((root_node.start_point, root_node.end_point), (0, root_node.type))
        if root_node.type == code:
            return [], states
        elif code in states:
            return [(code, idx, 'comesFrom', [code], states[code].copy())], states
        else:
            if root_node.type == 'identifier':
                states[code] = [idx]
            return [(code, idx, 'comesFrom', [], [])], states
            
    elif root_node.type in def_statement:
        name = root_node.child_by_field_name('name')
        value = root_node.child_by_field_name('value')
        DFG = []
        if value is None:
            indexs = tree_to_variable_index(name, index_to_code)
            for index in indexs:
                if index in index_to_code:
                    idx, code = index_to_code[index]
                    DFG.append((code, idx, 'comesFrom', [], []))
                    states[code] = [idx]
            return sorted(DFG, key=lambda x: x[1]), states
        else:
            name_indexs = tree_to_variable_index(name, index_to_code)
            value_indexs = tree_to_variable_index(value, index_to_code)
            temp, states = DFG_java(value, index_to_code, states)
            DFG += temp
            for index1 in name_indexs:
                if index1 in index_to_code:
                    idx1, code1 = index_to_code[index1]
                    for index2 in value_indexs:
                        if index2 in index_to_code:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'comesFrom', [code2], [idx2]))
                    states[code1] = [idx1]
            return sorted(DFG, key=lambda x: x[1]), states
            
    elif root_node.type in assignment:
        left_nodes = root_node.child_by_field_name('left')
        right_nodes = root_node.child_by_field_name('right')
        DFG = []
        temp, states = DFG_java(right_nodes, index_to_code, states)
        DFG += temp
        name_indexs = tree_to_variable_index(left_nodes, index_to_code)
        value_indexs = tree_to_variable_index(right_nodes, index_to_code)
        for index1 in name_indexs:
            if index1 in index_to_code:
                idx1, code1 = index_to_code[index1]
                for index2 in value_indexs:
                    if index2 in index_to_code:
                        idx2, code2 = index_to_code[index2]
                        DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in increment_statement:
        DFG = []
        indexs = tree_to_variable_index(root_node, index_to_code)
        for index1 in indexs:
            if index1 in index_to_code:
                idx1, code1 = index_to_code[index1]
                for index2 in indexs:
                    if index2 in index_to_code:
                        idx2, code2 = index_to_code[index2]
                        DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in if_statement:
        DFG = []
        current_states = states.copy()
        others_states = []
        flag = False
        tag = False
        if 'else' in root_node.type:
            tag = True
        for child in root_node.children:
            if 'else' in child.type:
                tag = True
            if child.type not in if_statement and flag is False:
                temp, current_states = DFG_java(child, index_to_code, current_states)
                DFG += temp
            else:
                flag = True
                temp, new_states = DFG_java(child, index_to_code, states)
                DFG += temp
                others_states.append(new_states)
        others_states.append(current_states)
        if tag is False:
            others_states.append(states)
        new_states = {}
        for dic in others_states:
            for key in dic:
                if key not in new_states:
                    new_states[key] = dic[key].copy()
                else:
                    new_states[key] += dic[key]
        for key in new_states:
            new_states[key] = sorted(list(set(new_states[key])))
        return sorted(DFG, key=lambda x: x[1]), new_states
        
    elif root_node.type in for_statement:
        DFG = []
        for child in root_node.children:
            temp, states = DFG_java(child, index_to_code, states)
            DFG += temp
        flag = False
        for child in root_node.children:
            if flag:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
            elif child.type == "local_variable_declaration":
                flag = True
        dic = {}
        for x in DFG:
            if (x[0], x[1], x[2]) not in dic:
                dic[(x[0], x[1], x[2])] = [x[3], x[4]]
            else:
                dic[(x[0], x[1], x[2])][0] = list(set(dic[(x[0], x[1], x[2])][0] + x[3]))
                dic[(x[0], x[1], x[2])][1] = sorted(list(set(dic[(x[0], x[1], x[2])][1] + x[4])))
        DFG = [(x[0], x[1], x[2], y[0], y[1]) for x, y in sorted(dic.items(), key=lambda t: t[0][1])]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in enhanced_for_statement:
        name = root_node.child_by_field_name('name')
        value = root_node.child_by_field_name('value')
        body = root_node.child_by_field_name('body')
        DFG = []
        for i in range(2):
            temp, states = DFG_java(value, index_to_code, states)
            DFG += temp
            name_indexs = tree_to_variable_index(name, index_to_code)
            value_indexs = tree_to_variable_index(value, index_to_code)
            for index1 in name_indexs:
                if index1 in index_to_code:
                    idx1, code1 = index_to_code[index1]
                    for index2 in value_indexs:
                        if index2 in index_to_code:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                    states[code1] = [idx1]
            temp, states = DFG_java(body, index_to_code, states)
            DFG += temp
        dic = {}
        for x in DFG:
            if (x[0], x[1], x[2]) not in dic:
                dic[(x[0], x[1], x[2])] = [x[3], x[4]]
            else:
                dic[(x[0], x[1], x[2])][0] = list(set(dic[(x[0], x[1], x[2])][0] + x[3]))
                dic[(x[0], x[1], x[2])][1] = sorted(list(set(dic[(x[0], x[1], x[2])][1] + x[4])))
        DFG = [(x[0], x[1], x[2], y[0], y[1]) for x, y in sorted(dic.items(), key=lambda t: t[0][1])]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in while_statement:
        DFG = []
        for i in range(2):
            for child in root_node.children:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
        dic = {}
        for x in DFG:
            if (x[0], x[1], x[2]) not in dic:
                dic[(x[0], x[1], x[2])] = [x[3], x[4]]
            else:
                dic[(x[0], x[1], x[2])][0] = list(set(dic[(x[0], x[1], x[2])][0] + x[3]))
                dic[(x[0], x[1], x[2])][1] = sorted(list(set(dic[(x[0], x[1], x[2])][1] + x[4])))
        DFG = [(x[0], x[1], x[2], y[0], y[1]) for x, y in sorted(dic.items(), key=lambda t: t[0][1])]
        return sorted(DFG, key=lambda x: x[1]), states
        
    else:
        DFG = []
        for child in root_node.children:
            if child.type in do_first_statement:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
        for child in root_node.children:
            if child.type not in do_first_statement:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
        return sorted(DFG, key=lambda x: x[1]), states

def clean_and_wrap_code(code_snippet):
    code_snippet = code_snippet.strip()
    if "class " in code_snippet and "{" in code_snippet:
        return code_snippet
    if any(x in code_snippet for x in ["public ", "private ", "protected "]):
        return f"public class DummyClass {{ {code_snippet} }}"
    return f"public class DummyClass {{ public void dummyMethod() {{ {code_snippet} }} }}"

def extract_dfg_from_code(code_snippet, parser):
    wrapped_code = clean_and_wrap_code(code_snippet)
    tree = parser.parse(bytes(wrapped_code, 'utf8'))
    root_node = tree.root_node
    
    tokens_index = tree_to_token_index(root_node)
    code_lines = [bytes(line, 'utf8') for line in wrapped_code.split('\n')]
    index_to_code = {}
    
    try:
        for idx, (start, end) in enumerate(tokens_index):
            val_bytes = index_to_code_token((start, end), code_lines)
            string_val = val_bytes.decode('utf8', errors='ignore') if isinstance(val_bytes, bytes) else val_bytes
            index_to_code[(start, end)] = ((start, end), string_val)
            
        DFG, _ = DFG_java(root_node, index_to_code, {})
        return DFG
    except Exception as e:
        return []

def extract_methods_from_file(file_path, parser):
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            code = f.read()
    except Exception as e:
        return []

    tree = parser.parse(bytes(code, 'utf8'))
    methods = []
    
    def traverse(node):
        if node.type in ['method_declaration', 'constructor_declaration']:
            start_byte = node.start_byte
            end_byte = node.end_byte
            code_bytes = bytes(code, 'utf8')
            method_code = code_bytes[start_byte:end_byte].decode('utf8', errors='ignore')
            if len(method_code.splitlines()) > 2 and "{" in method_code:
                methods.append({
                    "file": file_path,
                    "code": method_code
                })
        for child in node.children:
            traverse(child)
            
    traverse(tree.root_node)
    return methods


In [ ]:
# --- 2. MODEL UTILS ---

class ModelA(nn.Module):
    def __init__(self, encoder, config, tokenizer):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.tokenizer  = tokenizer
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None):
        ext_mask = (1.0 - attn_mask) * -10000.0
        ext_mask = ext_mask.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext_mask,
            head_mask=[None] * self.config.num_hidden_layers
        )[0]
        logits = self.classifier(self.dropout(out[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)
        return prob

def load_model(weights_path, device):
    print(f"Loading GraphCodeBERT from {weights_path}...")
    cfg = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
    cfg.num_labels = 2
    tok = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    enc = RobertaModel.from_pretrained("microsoft/graphcodebert-base", config=cfg)
    
    model = ModelA(enc, cfg, tok).to(device)
    try:
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model.eval()
        print("  ✓ Model loaded successfully")
        return model, tok
    except Exception as e:
        print(f"Error loading model weights: {e}")
        return None, None

def prepare_input(code, dfg, tokenizer, device="cpu", code_length=384, data_flow_length=128):
    total_len = code_length + data_flow_length
    code_lines = code.splitlines(keepends=True)
    tok = tokenizer(
        code, max_length=code_length, truncation=True,
        padding='max_length', return_offsets_mapping=True
    )
    input_ids, offsets = tok['input_ids'], tok['offset_mapping']
    
    dfg = dfg[:data_flow_length]
    dfg_ids = [tokenizer.unk_token_id] * len(dfg)
    
    def _char_idx(lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r, len(lines)))) + c

    n2t, p2n = {}, {}
    for ni, node in enumerate(dfg):
        sp, ep = node[1][0], node[1][1]
        pk     = (sp[0], sp[1], ep[0], ep[1])
        p2n[pk] = ni
        try:
            cs, ce = _char_idx(code_lines, sp), _char_idx(code_lines, ep)
            n2t[ni] = [i for i, (ts, te) in enumerate(offsets) 
                       if ts != te and ((ts >= cs and te <= ce) or (cs >= ts and ce <= te))]
        except IndexError:
            n2t[ni] = []

    mask = np.zeros((total_len, total_len), dtype=bool)
    mask[:code_length, :code_length] = True
    
    for ni, node in enumerate(dfg):
        ani = code_length + ni
        for ti in n2t.get(ni, []):
            mask[ani, ti] = mask[ti, ani] = True
        if len(node) > 4:
            for pp in node[4]:
                pk = (pp[0][0], pp[0][1], pp[1][0], pp[1][1])
                if pk in p2n:
                    ap = code_length + p2n[pk]
                    mask[ani, ap] = mask[ap, ani] = True
        mask[ani, ani] = True

    ids     = input_ids + dfg_ids
    p_ids   = [i + 2 for i in range(code_length)] + [0] * len(dfg_ids)
    
    pad = total_len - len(ids)
    if pad > 0:
        ids   += [tokenizer.pad_token_id] * pad
        p_ids += [1] * pad

    return {
        'input_ids': torch.tensor([ids], dtype=torch.long, device=device),
        'p_ids':     torch.tensor([p_ids], dtype=torch.long, device=device),
        'attn_mask': torch.tensor(np.array([mask]), dtype=torch.float, device=device)
    }

def infer_vulnerability(code, dfg, model, tokenizer, device, threshold=0.45, code_length=384):
    tok = tokenizer(code, return_offsets_mapping=True)
    total_tokens = len(tok['input_ids'])
    
    if total_tokens <= code_length:
        inputs = prepare_input(code, dfg, tokenizer, device=device, code_length=code_length)
        with torch.no_grad():
            prob = model(
                input_ids=inputs['input_ids'].to(device),
                p_ids=inputs['p_ids'].to(device), 
                attn_mask=inputs['attn_mask'].to(device)
            )
            p_vuln = prob[0][1].item()
        return p_vuln >= threshold, p_vuln

    stride = code_length // 2
    max_prob = 0.0
    code_lines = code.splitlines(keepends=True)
    
    def _get_string_for_tokens(start_idx, end_idx):
        try:
            start_char = tok['offset_mapping'][start_idx][0]
            end_char = tok['offset_mapping'][min(end_idx - 1, len(tok['offset_mapping']) - 1)][1]
            return code[start_char:end_char]
        except IndexError:
            return ""

    for i in range(0, total_tokens, stride):
        chunk_code = _get_string_for_tokens(i, i + code_length)
        if not chunk_code.strip(): continue
            
        chunk_dfg = [node for node in dfg if node[0] in chunk_code]
        inputs = prepare_input(chunk_code, chunk_dfg, tokenizer, device=device, code_length=code_length)
        
        with torch.no_grad():
            prob = model(
                input_ids=inputs['input_ids'].to(device),
                p_ids=inputs['p_ids'].to(device), 
                attn_mask=inputs['attn_mask'].to(device)
            )
            chunk_p = prob[0][1].item()
            if chunk_p > max_prob: max_prob = chunk_p
        if max_prob >= 0.95: break
            
    return max_prob >= threshold, max_prob

def infer_vulnerabilities_batched(methods, model, tokenizer, device, threshold=0.45, code_length=384, batch_size=32):
    all_chunks = []
    
    for m in methods:
        m_id, code, dfg, short_class = m
        tok = tokenizer(code, return_offsets_mapping=True, truncation=False)
        total_tokens = len(tok['input_ids'])
        
        if total_tokens <= code_length:
            all_chunks.append((m_id, code, dfg, short_class))
        else:
            stride = code_length // 2
            def _get_string_for_tokens(start_idx, end_idx):
                try:
                    start_char = tok['offset_mapping'][start_idx][0]
                    end_char = tok['offset_mapping'][min(end_idx - 1, len(tok['offset_mapping']) - 1)][1]
                    return code[start_char:end_char]
                except IndexError:
                    return ""
                    
            for i in range(0, total_tokens, stride):
                chunk_code = _get_string_for_tokens(i, i + code_length)
                if not chunk_code.strip(): continue
                chunk_dfg = [node for node in dfg if node[0] in chunk_code]
                all_chunks.append((m_id, chunk_code, chunk_dfg, short_class))
                
    results = {m[0]: 0.0 for m in methods}
    class_map = {m[0]: m[3] for m in methods}
    
    try:
        from tqdm import tqdm
        chunk_iterable = tqdm(range(0, len(all_chunks), batch_size), desc="Inferring in GPU batches", leave=True)
    except:
        chunk_iterable = range(0, len(all_chunks), batch_size)
        
    for i in chunk_iterable:
        batch_chunks = all_chunks[i:i+batch_size]
        batch_inputs = [prepare_input(c, d, tokenizer, device=device, code_length=code_length) for _, c, d, _ in batch_chunks]
        
        input_ids = torch.cat([inp['input_ids'] for inp in batch_inputs], dim=0)
        p_ids = torch.cat([inp['p_ids'] for inp in batch_inputs], dim=0)
        attn_mask = torch.cat([inp['attn_mask'] for inp in batch_inputs], dim=0)
        
        with torch.no_grad():
            probs = model(input_ids=input_ids, p_ids=p_ids, attn_mask=attn_mask)
            
        for idx, chunk_prob in enumerate(probs):
            m_id = batch_chunks[idx][0]
            p_vuln = chunk_prob[1].item()
            if p_vuln > results[m_id]:
                results[m_id] = p_vuln
                
    final_results = []
    for m_id, prob in results.items():
        is_vuln = prob >= threshold
        final_results.append((m_id, is_vuln, prob, class_map[m_id]))
        
    return final_results


In [ ]:
# --- 3. EXECUTION ---
JADX_VERSION = "1.4.7"
JADX_URL = f"https://github.com/skylot/jadx/releases/download/v{JADX_VERSION}/jadx-{JADX_VERSION}.zip"
TOOLS_DIR = Path("/kaggle/working/tools")
JADX_BIN = TOOLS_DIR / "jadx" / "bin" / "jadx"
OUT_DIR = Path("/kaggle/working/out")

def check_and_install_jadx():
    if shutil.which("jadx"):
        print("[*] JADX found in system PATH.")
        return "jadx"
    
    if JADX_BIN.exists():
        print(f"[*] Local JADX found at {JADX_BIN}")
        return str(JADX_BIN)
        
    print("[!] JADX not found. Downloading...")
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = TOOLS_DIR / "jadx.zip"
    
    urllib.request.urlretrieve(JADX_URL, zip_path)
    print("[*] Extracting JADX...")
    jadx_dir = TOOLS_DIR / "jadx"
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(jadx_dir)
        
    os.chmod(JADX_BIN, 0o755)
    zip_path.unlink()
    print("[*] JADX installed successfully.")
    return str(JADX_BIN)

def decompile_apk(jadx_path, apk_path, output_dir):
    print(f"[*] Decompiling {apk_path} to {output_dir}")
    os.makedirs(output_dir, exist_ok=True)
    cmd = [jadx_path, "--no-res", "-d", str(output_dir), str(apk_path)]
    try:
        subprocess.run(cmd, check=True, timeout=600, capture_output=True, text=True)
        print("[*] Decompilation finished.")
        return True
    except subprocess.TimeoutExpired:
        print("[!] Decompilation timed out.")
        return False
    except subprocess.CalledProcessError as e:
        print(f"[!] Decompilation failed: {e.stderr}")
        return False

def scan_directory_for_java(src_dir, target_package=None):
    java_files = []
    
    package_path = target_package.replace('.', os.sep) if target_package else None
    
    for root, _, files in os.walk(src_dir):
        if package_path and package_path not in root:
            continue
            
        for file in files:
            if file.endswith(".java"):
                java_files.append(os.path.join(root, file))
    return java_files

# =========================================================================
# KAGGLE PATH CONFIGURATION
# =========================================================================
# Point this to the root directory where all your APKs are uploaded
APK_DIR = Path("/kaggle/input/your-apks-folder/")

# Make sure to update your model path!
MODEL_PATH = Path("/kaggle/input/model2/model.bin")
THRESHOLD = 0.45

# NOTE: Set this to "auto" to dynamically parse the APK manifest
# Or set to e.g., 'com.stark.vpn' to explicitly filter. Set to None to scan everything.
TARGET_PACKAGE = "auto"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Using device: {device}")

# Run pipeline
if not APK_DIR.exists():
    print(f"[!] Error: Could not find APK directory at {APK_DIR}. Did you upload it?")
elif not MODEL_PATH.exists():
    print(f"[!] Error: Could not find model at {MODEL_PATH}. Make sure to update the path above.")
else:
    apk_files = list(APK_DIR.glob("*.apk"))
    print(f"[*] Found {len(apk_files)} APKs in {APK_DIR}")
    
    jadx_cmd = check_and_install_jadx()
    
    print("[*] Setting up Tree-sitter parsers...")
    java_lang, ts_parser = setup_tree_sitter()
    
    master_summary = []
    
    if ts_parser:
        model, tokenizer = load_model(MODEL_PATH, device)
        if model:
            for apk_path in apk_files:
                print(f"\n{'='*60}")
                print(f"[*] Analyzing: {apk_path.name}")
                print(f"{'='*60}")
                
                target_out_dir = OUT_DIR / apk_path.stem
                report_path = OUT_DIR / f"{apk_path.stem}_vuln_report.json"
                
                if report_path.exists():
                    print(f"[*] Skipping {apk_path.name}, report already exists.")
                    import json
                    try:
                        with open(report_path, "r") as f:
                            data = json.load(f)
                            t_scanned = data.get("total_functions_scanned", 0)
                            v_count = data.get("vulnerable_functions_count", 0)
                            master_summary.append({
                                "apk_name": apk_path.name,
                                "total_functions": t_scanned,
                                "vulnerable_functions": v_count
                            })
                            print("-" * 50)
                            print(f"Summary for {apk_path.name}")
                            print(f"  Total Functions Scanned : {t_scanned}")
                            print(f"  Vulnerable Functions    : {v_count}")
                            print(f"  Safe Functions          : {t_scanned - v_count}")
                            print("-" * 50)
                    except: pass
                    continue
                
                active_target_pkg = TARGET_PACKAGE
                if active_target_pkg == "auto":
                    try:
                        from androguard.core.bytecodes.apk import APK
                        a = APK(str(apk_path))
                        active_target_pkg = a.get_package()
                        print(f"[*] Auto-detected Target Package: {active_target_pkg}")
                    except Exception as e:
                        print(f"[!] Warning: Failed to parse package with androguard: {e}. Falling back to scanning all packages.")
                        active_target_pkg = None
                elif active_target_pkg:
                    print(f"[*] Explicit Target Package Filter: {active_target_pkg}")
                
                target_out_dir = OUT_DIR / apk_path.stem
                if decompile_apk(jadx_cmd, apk_path, target_out_dir):
                    java_files = scan_directory_for_java(target_out_dir, active_target_pkg)
                    print(f"[*] Found {len(java_files)} Java files inside package to analyze.")

                    total_vuln = 0
                    total_safe = 0

                    print("-" * 50)
                    print("VULNERABILITY REPORT")
                    print("-" * 50)
                    
                    try:
                        from tqdm import tqdm
                        file_iterable = tqdm(java_files, desc="Parsing AST & DFG", leave=True)
                    except ImportError:
                        file_iterable = java_files

                    all_methods = []
                    m_id_counter = 0

                    for j_file in file_iterable:
                        methods = extract_methods_from_file(j_file, ts_parser)
                        for m in methods:
                            code_text = m['code']
                            dfg = extract_dfg_from_code(code_text, ts_parser)
                            if not dfg: continue
                            
                            short_class = Path(j_file).stem
                            all_methods.append((m_id_counter, code_text, dfg, short_class))
                            m_id_counter += 1
                            
                    print(f"[*] Extracted {len(all_methods)} Java methods for GPU Inference.")

                    total_vuln = 0
                    total_safe = 0

                    print("-" * 50)
                    print("VULNERABILITY REPORT")
                    print("-" * 50)

                    report_data = {
                        "apk_name": apk_path.name,
                        "target_package": active_target_pkg,
                        "total_functions_scanned": len(all_methods),
                        "vulnerable_functions": [],
                        "all_probabilities": [],
                        "vulnerable_functions_count": 0,
                        "safe_functions_count": 0
                    }

                    if len(all_methods) > 0:
                        results = infer_vulnerabilities_batched(all_methods, model, tokenizer, device, THRESHOLD, batch_size=32)

                        for m_id, is_vuln, prob, short_class in results:
                            report_data["all_probabilities"].append(round(prob, 4))
                            if is_vuln:
                                total_vuln += 1
                                print(f"[!] VULNERABLE (Prob: {prob:.4f}) -> {short_class}.java")
                                report_data["vulnerable_functions"].append({
                                    "class": short_class,
                                    "probability": round(prob, 4)
                                })
                            else:
                                total_safe += 1
                                
                        report_data["vulnerable_functions_count"] = total_vuln
                        report_data["safe_functions_count"] = total_safe
                        
                    with open(report_path, "w") as f:
                        json.dump(report_data, f, indent=4)
                    print(f"[*] Saved detailed vulnerability report to: {report_path}")
                    
                    master_summary.append({
                        "apk_name": apk_path.name,
                        "total_functions": report_data["total_functions_scanned"],
                        "vulnerable_functions": report_data["vulnerable_functions_count"]
                    })

                    print("-" * 50)
                    print(f"Summary for {apk_path.name}")
                    print(f"  Total Functions Scanned : {total_vuln + total_safe}")
                    print(f"  Vulnerable Functions    : {total_vuln}")
                    print(f"  Safe Functions          : {total_safe}")
                    print("-" * 50)
                    
            if master_summary:
                import csv
                csv_path = OUT_DIR / "master_summary.csv"
                with open(csv_path, "w", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=["apk_name", "total_functions", "vulnerable_functions"])
                    writer.writeheader()
                    for row in master_summary:
                        writer.writerow(row)
                print(f"\n[*] Pipeline completely finished! Master CSV summary saved to: {csv_path}")
